# ETL Silver ➜ Gold FIFA 21 Players

Este notebook realiza a transformação dos dados da camada **Silver**
para a camada **Gold**, aplicando **modelagem dimensional (Star Schema)**.

### Tabelas Gold geradas:
- dim_ply (Jogador)
- dim_tm (Time)
- dim_pos (Posição)
- fat_ply_stats (Fato de atributos e ratings)


## Imports e Configurações

In [26]:
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

## Carrega variáveis de ambiente (.env)

In [27]:
load_dotenv("../.env")

DB_CONFIG = {
    'host': os.getenv('POSTGRES_HOST'),
    'port': int(os.getenv('POSTGRES_PORT')),
    'database': os.getenv('POSTGRES_DB'),
    'user': os.getenv('POSTGRES_USER'),
    'password': os.getenv('POSTGRES_PASSWORD')
}

DATABASE_URL = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL)

## Preparação da Camada Gold (DDL)

Antes de qualquer transformação ou carga, executamos o **DDL da camada Gold**.

Esse DDL é responsável por:
- Criar o schema `gold`
- Criar tabelas dimensão e fato
- Definir PKs, FKs, índices e comentários

Essa etapa garante:
- Integridade referencial
- Performance
- Padronização do modelo

In [28]:
DDL_PATH = "../Data Layer/gold/ddl.sql"

with open(DDL_PATH, "r") as file:
    ddl_sql = file.read()

with engine.begin() as conn:
    conn.execute(text(ddl_sql))

print("DDL da camada Gold executado com sucesso.")


DDL da camada Gold executado com sucesso.


## Leitura da Silver

Nesta etapa, os dados são lidos da camada Silver.

A Silver já contém dados:
- Limpos
- Tipados
- Padronizados

In [29]:
QUERY_SILVER = """
SELECT *
FROM silver.fifa21_players
"""

df_silver = pd.read_sql(QUERY_SILVER, engine)

df_silver.head()

,player_id,long_name,name,nationality,positions,age,overall_rating,potential_rating,team,contract_start_year,...,attack_work_rate,defense_work_rate,international_reputation,pace,shooting,passing,dribbling_stat,defending_stat,physical,hits
0,158023,Lionel Messi,L. Messi,Argentina,RW ST CF,33,93,93,FC Barcelona,2004.0,...,Medium,Low,5,85,92,91,95,38,65,372
1,20801,C. Ronaldo dos Santos Aveiro,Cristiano Ronaldo,Portugal,ST LW,35,92,92,Juventus,2018.0,...,High,Low,5,89,93,81,89,35,77,344
2,200389,Jan Oblak,J. Oblak,Slovenia,GK,27,91,93,Atlético Madrid,2014.0,...,Medium,Medium,3,87,92,78,90,52,90,86
3,192985,Kevin De Bruyne,K. De Bruyne,Belgium,CAM CM,29,91,91,Manchester City,2015.0,...,High,High,4,76,86,93,88,64,78,163
4,190871,Neymar da Silva Santos Jr.,Neymar Jr,Brazil,LW CAM,28,91,91,Paris Saint-Germain,2017.0,...,High,Medium,5,91,85,86,94,36,59,273


## Dimensões

As dimensões representam entidades descritivas do negócio.
Neste projeto:
- Jogador
- Time / Contrato
- Posição

Cada dimensão recebe uma **surrogate key**
utilizada posteriormente na tabela fato.

### Dimensão Jogador (dim_ply)

Contém informações demográficas e técnicas do jogador.

Cada jogador aparece **uma única vez** na dimensão.

In [30]:
dim_ply = (
    df_silver[[
        "player_id",
        "long_name",
        "name",
        "age",
        "height_cm",
        "weight_kg",
        "preferred_foot",
        "weak_foot",
        "skill_moves",
        "international_reputation",
        "nationality"
    ]]
    .drop_duplicates()
)

dim_ply.insert(0, "ply_key", range(1, len(dim_ply) + 1))

### Dimensão Time (dim_tm)

Representa o vínculo contratual do jogador com um time.

Nesta modelagem cada combinação de contrato é única.

In [31]:
dim_tm = (
    df_silver[[
        "team",
    ]]
    .drop_duplicates()
)

dim_tm.insert(0, "tm_key", range(1, len(dim_tm) + 1))

### Dimensão Posição (dim_pos)

Contém as posições jogáveis do atleta e sua posição principal.

In [32]:
dim_pos = (
    df_silver[[
        "positions",
        "best_position"
    ]]
    .drop_duplicates()
)

dim_pos.insert(0, "pos_key", range(1, len(dim_pos) + 1))

## Tabela Fato

A tabela fato concentra:
- Métricas
- Ratings
- Atributos quantitativos

Ela referencia as dimensões por meio de **foreign keys (surrogate keys)**.

### Mapeamento de Chaves (Lookups)

Aqui realizamos os joins entre a Silver e as dimensões
para substituir chaves naturais por surrogate keys.

In [33]:
df_fact = df_silver.copy()

df_fact = df_fact.merge(
    dim_ply[["ply_key", "player_id"]],
    on="player_id",
    how="left"
)

df_fact = df_fact.merge(
    dim_tm[["tm_key", "team"]],
    on=["team"],
    how="left"
)

df_fact = df_fact.merge(
    dim_pos[["pos_key", "positions", "best_position"]],
    on=["positions", "best_position"],
    how="left"
)


### Construção da fat_ply_stats

Seleciona apenas as colunas relevantes para análise
e renomeia as chaves para o padrão SRK (Surrogate Key).

In [ ]:
fat_ply_stats = df_fact[[
    "ply_key",
    "tm_key",
    "pos_key",

    "overall_rating",
    "potential_rating",
    "best_overall_rating",
    "growth",
    "total_stats",
    "base_stats",
    "hits",

    "value_eur",
    "wage_eur",
    "release_clause_eur",

    "contract_start_year",
    "contract_end_year",
    "joined_date",

    "pace",
    "shooting",
    "passing",
    "dribbling_stat",
    "defending_stat",
    "physical",

    "attacking_total",
    "crossing",
    "finishing",
    "heading_accuracy",
    "short_passing",
    "volleys",

    "skill_total",
    "dribbling",
    "curve",
    "fk_accuracy",
    "long_passing",
    "ball_control",

    "movement_total",
    "acceleration",
    "sprint_speed",
    "agility",
    "reactions",
    "balance",

    "power_total",
    "shot_power",
    "jumping",
    "stamina",
    "strength",
    "long_shots",

    "mentality_total",
    "aggression",
    "interceptions",
    "positioning",
    "vision",
    "penalties",
    "composure",

    "defending_total",
    "marking",
    "standing_tackle",
    "sliding_tackle",

    "goalkeeping_total",
    "gk_diving",
    "gk_handling",
    "gk_kicking",
    "gk_positioning",
    "gk_reflexes"
]].rename(columns={
    "ply_key": "ply_srk",
    "tm_key": "tm_srk",
    "pos_key": "pos_srk"
})

fat_ply_stats.insert(0, "fts_key", range(1, len(fat_ply_stats) + 1))

KeyError: "['joined_datepace'] not in index"

### Validações Básicas

Garante que todos os registros da fato
possuem relacionamento válido com as dimensões.

In [ ]:
assert fat_ply_stats["ply_srk"].isnull().sum() == 0
assert fat_ply_stats["tm_srk"].isnull().sum() == 0
assert fat_ply_stats["pos_srk"].isnull().sum() == 0

## Escrita das tabelas Gold no banco

As tabelas são carregadas no schema `gold`.

Utilizamos `append` pois o schema já foi criado via DDL.

In [ ]:
dim_ply.to_sql(
    "dim_ply",
    engine,
    schema="gold",
    if_exists="append",
    index=False
)

dim_tm.to_sql(
    "dim_tm",
    engine,
    schema="gold",
    if_exists="append",
    index=False
)

dim_pos.to_sql(
    "dim_pos",
    engine,
    schema="gold",
    if_exists="append",
    index=False
)

fat_ply_stats.to_sql(
    "fat_ply_stats",
    engine,
    schema="gold",
    if_exists="append",
    index=False
)


ProgrammingError: (psycopg2.errors.UndefinedColumn) column "contract_start_year" of relation "dim_tm" does not exist
LINE 1: INSERT INTO gold.dim_tm (tm_key, team, contract_start_year, ...
                                               ^

[SQL: INSERT INTO gold.dim_tm (tm_key, team, contract_start_year, contract_end_year, joined_date) VALUES (%(tm_key__0)s, %(team__0)s, %(contract_start_year__0)s, %(contract_end_year__0)s, %(joined_date__0)s), (%(tm_key__1)s, %(team__1)s, %(contract_start_y ... 113197 characters truncated ... 99)s, %(team__999)s, %(contract_start_year__999)s, %(contract_end_year__999)s, %(joined_date__999)s)]
[parameters: {'contract_end_year__0': 2021.0, 'tm_key__0': 1, 'contract_start_year__0': 2004.0, 'team__0': 'FC Barcelona', 'joined_date__0': datetime.datetime(2004, 7, 1, 0, 0), 'contract_end_year__1': 2022.0, 'tm_key__1': 2, 'contract_start_year__1': 2018.0, 'team__1': 'Juventus', 'joined_date__1': datetime.datetime(2018, 7, 10, 0, 0), 'contract_end_year__2': 202

## Exportação CSV da Gold Fato

Exportação opcional da tabela fato para uso externo,
validação local.

In [ ]:
OUTPUT_DIR = "../Data Layer/gold"
OUTPUT_FILE_CSV = "fifa21_gold.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_CSV)

fat_ply_stats.to_csv(output_path, index=False)

print(f"CSV Gold gerado em: {output_path}")

CSV Gold gerado em: ../Data Layer/gold/fifa21_gold.csv


## Validação pós-carga

Verificação simples para garantir que os dados
foram persistidos corretamente na camada Gold.

In [ ]:
pd.read_sql(
    "SELECT COUNT(*) FROM gold.fat_ply_stats",
    engine
)

,count
0,52478
